# pymoo NSGA-II — FULL-DATA, AMP, Higher Trial Budget

**MOML Assignment** — Prof. Aswin Kannan · Niranjan Gopal, Divyam Sareen · Due 2026-04-28

Same 10-D mixed-variable search space and same 3 objectives as `pymoo_optimization.ipynb`, but tuned to exploit a heavier GPU:

- **`train_subset_size = None`** → trains on the **full** Fashion-MNIST training set (60 000 images) every trial instead of an 8 K subset.
- **`use_amp = True`** → fp16 mixed-precision training via `torch.amp.autocast` + `GradScaler`. ~2× faster on Ampere/Ada GPUs (3070 / 4070 / A10G / L4 / 4090). Falls back to fp32 silently if the device is not CUDA.
- **Bigger trial budget**: `pop_size = 40 × n_gen = 5 = 200` evaluations (vs 80 in the lite notebook).
- **Lighter inference timing**: 10 warmup + 100 timed CPU passes (down from 50 + 500). Variance is still well under 1 ms; saves ~10 min over the study.

Wall-clock target on a real GPU (T4 / A10G / 4090): **~25–40 min total**, vs ~80 min for the 80-trial lite notebook on the same hardware.

Search space, objective contract, CSV schema, and analyzer are identical to the lite notebook so the two studies are directly comparable on the report's MOO-quality metrics.

In [ ]:
# --- Imports & path setup -------------------------------------------------
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from data_loader import DEVICE, PROJECT_ROOT, get_dataset_info
from moo_pymoo import run_pymoo_study
from analyze_study import analyze

print(f"Device       : {DEVICE}")
print(f"Project root : {PROJECT_ROOT}")
if str(DEVICE) == "cuda":
    import torch
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Configuration

| Knob | Default | Notes |
|---|---|---|
| `DATASET` | `fashion_mnist` | Same as the lite notebook for direct comparison. |
| `POP_SIZE`, `N_GEN` | 40, 5 | 200 total evaluations (2.5× the lite notebook). |
| `TRAIN_SUBSET_SIZE` | `None` | Full dataset (60 000 images for FMNIST). |
| `USE_AMP` | `True` | fp16 autocast + GradScaler — ignored on CPU. |
| `INFERENCE_WARMUP`, `INFERENCE_TIMED` | 10, 100 | CPU-batch=1 timing convention preserved, just fewer passes. |
| `NUM_WORKERS` | 4 | Bump to 4 if your storage and CPU can keep up; drop to 0 on Windows if you hit issues. |
| `SEED` | 42 | Per-trial seed = `SEED + trial_index`. |

In [ ]:
# --- Configuration --------------------------------------------------------
DATASET           = "fashion_mnist"
POP_SIZE          = 40
N_GEN             = 5
TRAIN_SUBSET_SIZE = None        # None = full training set
USE_AMP           = True        # fp16 mixed precision (CUDA only)
INFERENCE_WARMUP  = 10
INFERENCE_TIMED   = 100
NUM_WORKERS       = 4
SEED              = 42

COMPARE_PARETO    = None        # set to a botorch pareto_front.csv for joint-front overlay

info = get_dataset_info(DATASET)
total_trials = POP_SIZE * N_GEN
print(f"Dataset        : {DATASET}  (classes={info['num_classes']}, native res={info['default_resolution']})")
print(f"Total trials   : {total_trials} (pop={POP_SIZE} × gen={N_GEN})")
print(f"Train subset   : {'FULL DATASET' if TRAIN_SUBSET_SIZE is None else f'{TRAIN_SUBSET_SIZE:,} samples'}")
print(f"AMP            : {USE_AMP and str(DEVICE)=='cuda'}")

## Full optimization run

Streams every trial to `trials.csv` as it completes. Expected wall-clock on a real GPU:

| GPU | per-trial | 200 trials |
|---|---|---|
| GTX 1650 | ~30 s | ~100 min |
| T4 / Colab | ~10 s | ~33 min |
| A10G / L4 | ~6 s | ~20 min |
| RTX 4090 | ~3-4 s | ~12 min |

Output dir: `results/pymoo/<dataset>/pymoo_nsga2_<dataset>_<timestamp>/`.

In [ ]:
summary, study_dir = run_pymoo_study(
    dataset_name=DATASET,
    pop_size=POP_SIZE,
    n_gen=N_GEN,
    train_subset_size=TRAIN_SUBSET_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
    inference_warmup=INFERENCE_WARMUP,
    inference_timed=INFERENCE_TIMED,
)

print("\n" + "=" * 70)
print(f"Study dir : {study_dir}")
print(f"Wall time : {summary['elapsed_seconds']/60:.1f} min")
print(f"Pareto    : {summary['n_pareto_points']} non-dominated points (final population)")

## Analysis — Pareto metrics, table, plots, appendix solution

`analyze(study_dir)` is framework-agnostic and produces four plots: 2D-panel staircase, parallel coordinates, 2D bubble (`plot_3d_scatter.png`), and the MATLAB-style 3D Pareto (`plot_3d_pareto.png`).

In [ ]:
compare_path = Path(COMPARE_PARETO) if COMPARE_PARETO else None
metrics = analyze(study_dir, compare_pareto_path=compare_path)

print(f"n_trials       : {metrics['n_trials']}")
print(f"n_pareto       : {metrics['n_pareto_points']}")
print(f"hypervolume    : {metrics['hypervolume']:,.2f}")
print(f"spacing        : {metrics['spacing']:.4f}")
print(f"acc max        : {metrics['extremes']['accuracy_max']:.4f}")
print(f"ms  min        : {metrics['extremes']['inference_ms_min']:.4f}")
print(f"params min     : {metrics['extremes']['param_count_min']:,}")
if 'comparison' in metrics:
    c = metrics['comparison']
    print("\n--- vs comparison front ---")
    print(f"joint pareto      : {c['joint_n_pareto']}")
    print(f"ours surviving    : {c['ours_surviving_in_joint']}")
    print(f"other surviving   : {c['other_surviving_in_joint']}")
    print(f"GD ours -> joint  : {c['gd_ours_to_joint']:.6f}")
    print(f"GD other-> joint  : {c['gd_other_to_joint']:.6f}")

### Pareto table (4 representative solutions)

In [ ]:
pareto_table = pd.read_csv(study_dir / "pareto_table.csv")
pareto_table

### 2D panels — pairwise trade-offs with Pareto staircase

In [ ]:
display(Image(filename=str(study_dir / "plot_2d_panels.png")))

### Parallel coordinates of the Pareto front

In [ ]:
display(Image(filename=str(study_dir / "plot_parallel_coords.png")))

### 2D bubble plot — three objectives encoded in size + color

In [ ]:
display(Image(filename=str(study_dir / "plot_3d_scatter.png")))

### 3D Pareto-front scatter (MATLAB-style)

In [ ]:
display(Image(filename=str(study_dir / "plot_3d_pareto.png")))

## Appendix solution (balanced pick)

In [ ]:
with open(study_dir / "appendix_solution.json") as f:
    appendix = json.load(f)

print(f"Label              : {appendix['label']}")
print(f"Trial number       : {appendix.get('trial_number')}")
print()
print("--- Objectives ---")
for k, v in appendix['objectives'].items():
    print(f"  {k:<14s} : {v}")
print()
print("--- Decision variables ---")
for k, v in appendix['decision_variables'].items():
    print(f"  {k:<18s} : {v}")

## Compare against BoTorch (run after `botorch_optimization_full.ipynb` finishes)

In [ ]:
candidate = PROJECT_ROOT / "results" / "botorch" / DATASET
if candidate.exists():
    runs = sorted([p for p in candidate.iterdir() if p.is_dir()])
    if runs:
        latest = runs[-1] / "pareto_front.csv"
        print(f"botorch latest pareto: {latest}")
        print(f"\nTo enable comparison, set in the config cell:\n  COMPARE_PARETO = r\"{latest}\"")
    else:
        print(f"(no botorch runs yet in {candidate})")
else:
    print(f"(no botorch results for dataset={DATASET})")

## Upload results to a private HF Dataset

HF Jobs containers are ephemeral — the filesystem evaporates when the timeout fires or you `hf jobs stop`. Push `study_dir` to a private HF Dataset so the trials, Pareto, summary JSON, and plots survive.

Set `HF_DATASET_REPO` to enable. The Job needs to have been launched with `--secrets HF_TOKEN`; if so, the token is in the env and login is automatic. The repo is created with `exist_ok=True`, so the same dataset can hold multiple studies (each lands in its own subfolder under `pymoo/`).

In [ ]:
HF_DATASET_REPO = None    # e.g. "kvelidanda/moml-results"  -- set to enable upload

if HF_DATASET_REPO:
    import os
    from huggingface_hub import HfApi, login

    if "HF_TOKEN" in os.environ:
        login(token=os.environ["HF_TOKEN"])

    api = HfApi()
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(
        folder_path=str(study_dir),
        path_in_repo=f"pymoo/{study_dir.name}",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        commit_message=f"pymoo NSGA-II study: {study_dir.name}",
    )
    print(f"Uploaded → https://huggingface.co/datasets/{HF_DATASET_REPO}/tree/main/pymoo/{study_dir.name}")
else:
    print("Upload skipped. Set HF_DATASET_REPO above to enable.")